# Model Evaluation — Team11 Joel Puthenparambil

Evaluates all three model outputs using **BERTScore** (`xlm-roberta-base`).

Reference answers come from the course dataset Google Sheet. Each model's answers are compared against these.

Upload the dataset xlsx and the three result CSVs when prompted.

In [3]:
# Install evaluation library
!pip install bert-score pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00


In [4]:
# Imports
import pandas as pd
from bert_score import score as bert_score

## Load files

Upload:
- `dataset.xlsx` — the course Google Sheet downloaded as Excel (contains ground truth answers)
- `model1_inference.csv`
- `model2_finetuned.csv`
- `model3_rag.csv`

In [5]:
# Upload files (Colab)
from google.colab import files
uploaded = files.upload()

Saving model3_rag.csv to model3_rag.csv
Saving model2_finetuned.csv to model2_finetuned.csv
Saving model1_inference.csv to model1_inference.csv
Saving Austrian Tax Law Dataset.xlsx to Austrian Tax Law Dataset.xlsx


In [6]:
# Load ground truth from the 'Dataset' sheet, column 'correct_answer'
gt = pd.read_excel('Austrian Tax Law Dataset.xlsx', sheet_name='Dataset')
print("Columns:", gt.columns.tolist())
print("Rows:", len(gt))
gt.head(2)

Columns: ['id', 'prompt', 'correct_answer', 'sources', 'student_id', 'notes', 'corrected sources', 'direct citations from sources']
Rows: 645


,id,prompt,correct_answer,sources,student_id,notes,corrected sources,direct citations from sources
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...,"Das Einkommen, das der unbeschränkt Steuerpfli...",§ 7 Abs. 1 KStG 1988,h12239827,§ 7 Abs. 1 KStG 1988 allein ist nicht ausreich...,§ 7 Abs. 1 KStG\n§ 7 Abs. 2 KStG,"""Der Körperschaftsteuer ist das Einkommen zugr..."
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ...",Die nicht verrechneten Zinsen stellen eine ver...,§ 7 Abs. 2/3 KStG 1988; § 8 Abs. 2 KStG 1988; ...,h12239827,NaN,NaN,NaN


In [7]:
# Column names confirmed from the actual file
ID_COL     = 'id'             # question ID
ANSWER_COL = 'correct_answer' # ground truth answer column

gt = gt[[ID_COL, ANSWER_COL]].rename(columns={ANSWER_COL: 'gold'})
gt = gt.dropna(subset=['gold'])  # drop rows with no reference answer
print(f"Ground truth rows: {len(gt)}")
gt.head(2)

Ground truth rows: 645


,id,gold
0,CORP-TAX-001,"Das Einkommen, das der unbeschränkt Steuerpfli..."
1,CORP-TAX-002,Die nicht verrechneten Zinsen stellen eine ver...


In [8]:
# Load model outputs and merge with ground truth on id
m1 = pd.read_csv('model1_inference.csv')
m2 = pd.read_csv('model2_finetuned.csv')
m3 = pd.read_csv('model3_rag.csv')

df = gt.copy()
df = df.merge(m1[['id','answer']].rename(columns={'answer':'model1'}), on='id', how='inner')
df = df.merge(m2[['id','answer']].rename(columns={'answer':'model2'}), on='id', how='inner')
df = df.merge(m3[['id','answer']].rename(columns={'answer':'model3'}), on='id', how='inner')
df = df.fillna('')

print(f"Questions with all answers: {len(df)}")
df.head(2)

Questions with all answers: 643


,id,gold,model1,model2,model3
0,CORP-TAX-001,"Das Einkommen, das der unbeschränkt Steuerpfli...",Die Steuerbemessungsgrundlage für die Körpersc...,Die steuerliche Bemessungsgrundlage für die Kö...,Die steuerliche Bemessungsgrundlage für die Kö...
1,CORP-TAX-002,Die nicht verrechneten Zinsen stellen eine ver...,Die Vergabe eines zinslosen Darlehens einer Kö...,Die steuerliche Konsequence ist: Einem Unterne...,Die Vergabe eines zinslosen Darlehens durch ei...


## BERTScore — each model vs ground truth

In [9]:
refs = df['gold'].tolist()

# Model 1
print("Computing BERTScore: Model 1 vs ground truth ...")
P1, R1, F1_1 = bert_score(
    df['model1'].tolist(), refs,
    model_type='xlm-roberta-base', lang='de', verbose=False
)
print(f"  Precision: {P1.mean().item():.4f}  Recall: {R1.mean().item():.4f}  F1: {F1_1.mean().item():.4f}")

# Model 2
print("Computing BERTScore: Model 2 vs ground truth ...")
P2, R2, F1_2 = bert_score(
    df['model2'].tolist(), refs,
    model_type='xlm-roberta-base', lang='de', verbose=False
)
print(f"  Precision: {P2.mean().item():.4f}  Recall: {R2.mean().item():.4f}  F1: {F1_2.mean().item():.4f}")

# Model 3
print("Computing BERTScore: Model 3 vs ground truth ...")
P3, R3, F1_3 = bert_score(
    df['model3'].tolist(), refs,
    model_type='xlm-roberta-base', lang='de', verbose=False
)
print(f"  Precision: {P3.mean().item():.4f}  Recall: {R3.mean().item():.4f}  F1: {F1_3.mean().item():.4f}")

Computing BERTScore: Model 1 vs ground truth ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Precision: 0.8306  Recall: 0.8193  F1: 0.8246
Computing BERTScore: Model 3 vs ground truth ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Precision: 0.8515  Recall: 0.8688  F1: 0.8597


## Summary table

In [10]:
# Combine results and save
summary = pd.DataFrame([
    {
        'model': 'Model 1 (Mistral-7B inference)',
        'BERTScore_Precision': round(P1.mean().item(), 4),
        'BERTScore_Recall':    round(R1.mean().item(), 4),
        'BERTScore_F1':        round(F1_1.mean().item(), 4),
    },
    {
        'model': 'Model 2 (TinyLlama fine-tuned)',
        'BERTScore_Precision': round(P2.mean().item(), 4),
        'BERTScore_Recall':    round(R2.mean().item(), 4),
        'BERTScore_F1':        round(F1_2.mean().item(), 4),
    },
    {
        'model': 'Model 3 (RAG + Gemini)',
        'BERTScore_Precision': round(P3.mean().item(), 4),
        'BERTScore_Recall':    round(R3.mean().item(), 4),
        'BERTScore_F1':        round(F1_3.mean().item(), 4),
    },
])

summary.to_csv('evaluation_report.csv', index=False)
print("Saved evaluation_report.csv")
summary

Saved evaluation_report.csv


,model,BERTScore_Precision,BERTScore_Recall,BERTScore_F1
0,Model 1 (Mistral-7B inference),0.8372,0.8544,0.8454
1,Model 2 (TinyLlama fine-tuned),0.8306,0.8193,0.8246
2,Model 3 (RAG + Gemini),0.8515,0.8688,0.8597


In [11]:
# Download the report
files.download('evaluation_report.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>